# PersianQA: Gemma 3 4B QLoRA on Colab T4

**Single-file, standalone research notebook.** No local Python modules, uploaded ZIP files, or GitHub clone are required.

**Research question:** How does 4-bit LoRA fine-tuning change the performance of the same instruction-tuned Gemma 3 4B model on PersianQA?

- Base checkpoint: [`unsloth/gemma-3-4b-it-unsloth-bnb-4bit`](https://huggingface.co/unsloth/gemma-3-4b-it-unsloth-bnb-4bit)
- Dataset loaded **directly**: [`SajjadAyoubi/persian_qa`](https://huggingface.co/datasets/SajjadAyoubi/persian_qa)
- Hardware: a single Google Colab T4 (16 GB VRAM); 4-bit quantization + LoRA
- Held-out test: all official examples by default; train-only validation split is held out for possible future model selection; this pilot does not use it
- Metrics: Persian-normalized Exact Match, frequency-correct token F1, answerability scores, paired bootstrap 95% confidence intervals

**Important:** Google released Gemma **2** in 2B/9B/27B sizes, not 4B. This notebook switches to **Gemma 3 4B**, a *different* model family. Its scores must not be compared as a controlled before/after improvement over the original Gemma 2 9B notebook. The unmodified *Gemma 3 4B Instruct* checkpoint is the baseline for this experiment.

**No scores are embedded or fabricated.** Run all cells on a real T4 to obtain results. The default training budget is a **250-optimizer-step pilot**, not a full-dataset epoch. The full test is scored if `SMOKE_TEST=False`.

## 0 — Runtime and dependencies

In Colab: **Runtime → Change runtime type → T4 GPU**. Execute in order. The first installation may require a Colab runtime restart if it updates PyTorch; if so, rerun setup cells.

In [1]:
import os, sys, subprocess
assert 'google.colab' in sys.modules or os.path.exists('/content'), 'Open in Google Colab for the GPU experiment.'
# Installing Unsloth with its declared dependencies avoids mismatched no-deps installations.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth', 'datasets>=3.0,<5', 'huggingface_hub>=0.34', 'sentencepiece', 'protobuf'])
print('Package installation finished. If Colab requests a restart, restart and rerun setup.')

Package installation finished. If Colab requests a restart, restart and rerun setup.


In [2]:
import torch
assert torch.cuda.is_available(), 'CUDA unavailable. Select T4 GPU in Runtime settings.'
name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {name}; VRAM: {mem_gb:.1f} GiB; PyTorch: {torch.__version__}')
assert mem_gb >= 14, 'This configuration expects a GPU with roughly 16 GB VRAM.'

GPU: Tesla T4; VRAM: 14.6 GiB; PyTorch: 2.11.0+cu130


## 1 — Experiment settings

Use `SMOKE_TEST=True` only to check that the pipeline runs (those scores are **not** full-test research results). Keep it `False` to evaluate all official test examples. Set `MAX_STEPS` higher or use additional runs only after checking Colab runtime limits.

In [3]:
from pathlib import Path
import json, random, hashlib, unicodedata, re, gc, time, platform
from collections import Counter
from datetime import datetime, timezone
import numpy as np
from datasets import load_dataset, Dataset

SEED = 3407
MODEL_ID = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'
DATASET_ID = 'SajjadAyoubi/persian_qa'
SMOKE_TEST = False             # True: 32 test examples and 5 optimization steps
MAX_STEPS = 250               # T4 pilot run, NOT an entire training epoch
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LORA_R = 16                    # Lower than original rank 128 for T4 memory
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
MAX_SEQ_LENGTH = 896
MAX_CONTEXT_TOKENS = 560
MAX_NEW_TOKENS = 72
VALIDATION_FRACTION = 0.10
NO_ANSWER_TEXT = 'پاسخی در متن وجود ندارد'
BOOTSTRAP_REPEATS = 2000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Output location: Drive if mount succeeds, otherwise /content. No input files required.
BASE_OUTPUT = Path('/content/gemma3_persianqa_results')
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE_OUTPUT = Path('/content/drive/MyDrive/Gemma3_4B_PersianQA')
    except Exception as exc:
        print('Drive unavailable; writing locally instead:', exc)
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Output directory:', BASE_OUTPUT)
print('Mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL HELD-OUT TEST')

Mounted at /content/drive
Output directory: /content/drive/MyDrive/Gemma3_4B_PersianQA
Mode: FULL HELD-OUT TEST


## 2 — Directly download PersianQA from Hugging Face

The dataset is still `SajjadAyoubi/persian_qa`. Its legacy `persian_qa.py` script cannot be executed by modern `datasets`, so this section reads Hugging Face's **auto-converted Parquet files** from the same dataset repository instead. The original `train` and official held-out `validation` (source `pqa_test.json`) splits are preserved. We never use the official test data for LoRA training or hyperparameter selection. A deterministic passage-group split reserves an internal development set (not used for model selection in this pilot).


In [4]:
# Load Hugging Face's auto-converted Parquet files directly from the same dataset
# repository. Recent datasets versions no longer execute its legacy persian_qa.py.
# These files preserve the dataset's original train and official-test-as-validation splits.
PARQUET_ROOT = (
    f"https://huggingface.co/datasets/{DATASET_ID}/"
    "resolve/refs%2Fconvert%2Fparquet/persian_qa"
)
raw = load_dataset(
    "parquet",
    data_files={
        "train": f"{PARQUET_ROOT}/train/0000.parquet",
        "validation": f"{PARQUET_ROOT}/validation/0000.parquet",
    },
)
print(raw)
print('Available splits:', list(raw.keys()))
assert 'train' in raw, 'Expected a train split.'
OFFICIAL_TEST_SPLIT = 'test' if 'test' in raw else 'validation'
assert OFFICIAL_TEST_SPLIT in raw, 'Expected official held-out test/validation split.'
train_source, official_test = raw['train'], raw[OFFICIAL_TEST_SPLIT]
required = {'question', 'context', 'answers'}
assert required.issubset(train_source.column_names), train_source.column_names
assert required.issubset(official_test.column_names), official_test.column_names
assert len(train_source)>1000 and len(official_test)>100, 'Unexpected dataset length; verify dataset revision.'
print(f'Source train={len(train_source)}, official held-out ({OFFICIAL_TEST_SPLIT})={len(official_test)}')
print('Example keys:', train_source.column_names)
print('First answer example:', train_source[0]['answers'])

# Avoid putting questions from the same passage in both train and internal validation.
def passage_group(row):
    return hashlib.sha256((str(row.get('title','')) + '\n' + str(row['context'])).encode('utf-8')).hexdigest()

def is_internal_validation(row):
    # Stable hash-based group assignment independent of dataset row order.
    val = int(passage_group(row)[:8], 16) / 0x100000000
    return val < VALIDATION_FRACTION

train_indices = [i for i, row in enumerate(train_source) if not is_internal_validation(row)]
val_indices = [i for i, row in enumerate(train_source) if is_internal_validation(row)]
train_data = train_source.select(train_indices).shuffle(seed=SEED)
internal_val = train_source.select(val_indices)
assert len(train_data) > 0 and len(internal_val) > 0
assert set(passage_group(x) for x in train_data).isdisjoint(
       set(passage_group(x) for x in internal_val)), 'Passage leakage between train and dev.'
assert set(str(x['id']) for x in train_data).isdisjoint(
       set(str(x['id']) for x in internal_val)), 'ID overlap between train and dev.'
if SMOKE_TEST:
    train_data = train_data.select(range(min(96, len(train_data))))
    official_test = official_test.select(range(min(32, len(official_test))))
    MAX_STEPS = 5
print('Training rows:', len(train_data), 'internal validation rows:', len(internal_val),
      'official test rows to evaluate:', len(official_test))
assert not SMOKE_TEST or len(official_test) <= 32

persian_qa/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.62MB            

persian_qa/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/165k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 9008
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 930
    })
})
Available splits: ['train', 'validation']
Source train=9008, official held-out (validation)=930
Example keys: ['id', 'title', 'context', 'question', 'answers']
First answer example: {'text': ['در شرق شهر مبارکه'], 'answer_start': [114]}
Training rows: 8198 internal validation rows: 810 official test rows to evaluate: 930


## 3 — Load quantized Gemma 3 and its text tokenizer

Gemma 3 4B also supports images, but this study is deliberately **text-only**. `FastModel` provides Gemma 3's supported Unsloth loading path on an FP16-only T4. We use its Hugging Face text tokenizer for consistent chat prompts in training and evaluation.

In [5]:
# Import Unsloth BEFORE transformers to enable its patches.
from unsloth import FastModel
from transformers import AutoTokenizer
from huggingface_hub import HfApi

# Optional: add HF_TOKEN via Colab Secrets if model download requires gated access.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

model, model_processor = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
assert tokenizer.chat_template, 'Model tokenizer must supply its native chat template.'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
assert tokenizer.pad_token_id is not None
print('Model:', MODEL_ID)
print('Tokens:', 'PAD:', tokenizer.pad_token_id, 'EOS:', tokenizer.eos_token_id)
print('Model class:', model.__class__.__name__)
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
except Exception as exc:
    MODEL_REVISION = 'unavailable'
    print('Could not pin model revision:', exc)
try:
    DATASET_REVISION = HfApi().dataset_info(DATASET_ID).sha
except Exception as exc:
    DATASET_REVISION = 'unavailable'
    print('Could not pin dataset revision:', exc)
print('Model revision:', MODEL_REVISION)
print('Dataset revision:', DATASET_REVISION)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

## 4 — Unified PersianQA prompt construction and data integrity

The same prompt builder and context truncation are used for the baseline, training and final evaluation. Unanswerable examples are trained to produce one explicit abstention phrase. Long contexts are truncated from the right and that limitation is recorded; the full original passage is retained for reference scoring.

In [6]:
def answer_texts(row):
    answers = row['answers']
    # HF dataset has SQuAD-like {'text': [...], 'answer_start': [...]}.
    texts = answers.get('text', []) if isinstance(answers, dict) else []
    return [str(s).strip() for s in texts if s is not None and str(s).strip()]

def pick_training_answer(row):
    choices = answer_texts(row)
    return choices[0] if choices else NO_ANSWER_TEXT

def truncate_context(raw_context):
    ids = tokenizer.encode(str(raw_context), add_special_tokens=False)
    shortened = ids[:MAX_CONTEXT_TOKENS]
    return tokenizer.decode(shortened, skip_special_tokens=True), len(ids) > len(shortened)

def build_prompt(row):
    context, shortened = truncate_context(row['context'])
    question = str(row['question']).strip()
    def render(ctx):
        content = (
            'با توجه به متن زیر، فقط پاسخ کوتاه و دقیق پرسش را بنویس. '
            f'اگر پاسخ در متن وجود ندارد، دقیقا بنویس: {NO_ANSWER_TEXT}. '
            'از توضیح اضافی خودداری کن.\n\n'
            f'متن: {ctx}\n\nسؤال: {question}'
        )
        return tokenizer.apply_chat_template(
            [{'role':'user','content':content}], tokenize=False,
            add_generation_prompt=True,
        )
    prompt = render(context)
    # Guarantee max generation budget for every example, even with long Persian tokenization.
    if len(tokenizer.encode(prompt, add_special_tokens=False)) > MAX_SEQ_LENGTH - MAX_NEW_TOKENS:
        ids = tokenizer.encode(context, add_special_tokens=False)
        while len(ids) and len(tokenizer.encode(prompt, add_special_tokens=False)) > MAX_SEQ_LENGTH - MAX_NEW_TOKENS:
            ids = ids[:-32]
            context = tokenizer.decode(ids, skip_special_tokens=True)
            prompt = render(context)
            shortened = True
    assert len(tokenizer.encode(prompt, add_special_tokens=False)) <= MAX_SEQ_LENGTH - MAX_NEW_TOKENS
    return prompt, context, shortened

def training_item(row):
    prefix, _, _ = build_prompt(row)
    answer = pick_training_answer(row)
    # Native Gemma chat template determines the stop marker, no hardcoded ChatML/Alpaca tags.
    # The answer turn is assembled by the tokenizer-native full conversation template.
    user_section = prefix
    # Prefix already ends at the assistant generation start. Build the answer turn once.
    # For a Gemma3 template this ends with <end_of_turn>.
    content_end = tokenizer.apply_chat_template(
        [{'role':'user','content':'PLACEHOLDER_USER'}, {'role':'assistant','content':answer}],
        tokenize=False, add_generation_prompt=False,
    )
    placeholder_prefix = tokenizer.apply_chat_template(
        [{'role':'user','content':'PLACEHOLDER_USER'}],
        tokenize=False, add_generation_prompt=True,
    )
    assert content_end.startswith(placeholder_prefix), 'Unexpected chat template: answer does not follow assistant prefix.'
    assistant_suffix = content_end[len(placeholder_prefix):]
    prefix_ids = tokenizer.encode(user_section, add_special_tokens=False)
    target_ids = tokenizer.encode(assistant_suffix, add_special_tokens=False)
    # Never drop a reference answer mid-sentence. Skip oversized targets instead.
    if len(target_ids) > MAX_NEW_TOKENS:
        return None
    all_ids = prefix_ids + target_ids
    if len(all_ids) > MAX_SEQ_LENGTH or not target_ids:
        return None
    return {'input_ids': all_ids, 'attention_mask':[1] * len(all_ids),
            'labels':[-100]*len(prefix_ids) + target_ids}

# Inspect prompt format and verify no training loss is assigned to question/context.
example_prompt, example_context, example_truncated = build_prompt(train_data[0])
example_item = training_item(train_data[0])
assert example_item is not None and any(x != -100 for x in example_item['labels'])
assert example_item['labels'][0] == -100
print('Native prompt preview:', example_prompt[:650])
print('Truncated:', example_truncated, '| prompt tokens:',len(tokenizer.encode(example_prompt, add_special_tokens=False)))
print('Target token count:', sum(x != -100 for x in example_item['labels']))

Native prompt preview: <bos><start_of_turn>user
با توجه به متن زیر، فقط پاسخ کوتاه و دقیق پرسش را بنویس. اگر پاسخ در متن وجود ندارد، دقیقا بنویس: پاسخی در متن وجود ندارد. از توضیح اضافی خودداری کن.

متن: کارت‌بازی یا ورق‌بازی یا پاسوربازی هر نوع بازی است که از کارت‌های بازی به عنوان دستگاه اصلی که بازی با آن انجام می‌شود، است. تعداد بیشماری بازی با ورق وجود دارد. برگزاری مسابقات بین‌المللی به‌طور رسمی قوانین استانداردی دارند، اما بیشتر آنها بازی‌های مردمی هستند که قوانین آنها براساس منطقه، فرهنگ و شخص متفاوت است. یک بازی ورق با بسته کارت‌های بازی انجام می‌شود که از نظر اندازه و شکل یکسان هستند. هر کارت دو طرف دارد، صورت و پشت. به‌طور معمول پشت کارت‌ها قابل تشخیص نی
Truncated: False | prompt tokens: 341
Target token count: 21


## 5 — Correct PersianQA metrics (CPU-side)

Normalization unifies Arabic/Persian letters, digits, zero-width joiners, diacritics, punctuation and whitespace. F1 uses **token frequencies** (`Counter`), not unique-token sets. For unanswerable references, exact abstention gets EM=F1=1 and a nonempty answer gets zero. For multiple references, take the highest score across gold answers.

In [7]:
PERSIAN_TRANSLATION = str.maketrans({
    'ي':'ی', 'ى':'ی', 'ك':'ک', 'ۀ':'ه', 'ة':'ه', 'أ':'ا', 'إ':'ا', 'ٱ':'ا',
    '۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9',
    '٠':'0','١':'1','٢':'2','٣':'3','٤':'4','٥':'5','٦':'6','٧':'7','٨':'8','٩':'9'
})

def normalize_fa(text):
    text = unicodedata.normalize('NFKC', str(text)).translate(PERSIAN_TRANSLATION)
    text = ''.join(' ' if (unicodedata.category(c).startswith('P') or
                          unicodedata.category(c).startswith('C'))
                   else '' if unicodedata.category(c).startswith('M') else c
                   for c in text)
    return ' '.join(text.casefold().split())

NORMALIZED_ABSTENTION = normalize_fa(NO_ANSWER_TEXT)
def canonical_prediction(output):
    output = str(output).strip()
    # Only an exact normalized abstention phrase is counted as no-answer.
    return '' if normalize_fa(output) == NORMALIZED_ABSTENTION else output

def token_f1(prediction, reference):
    p, r = normalize_fa(prediction).split(), normalize_fa(reference).split()
    if not p and not r: return 1.0
    if not p or not r: return 0.0
    overlap = sum((Counter(p) & Counter(r)).values())
    return 2 * overlap / (len(p) + len(r))

def score_example(output, references):
    prediction = canonical_prediction(output)
    candidates = references if references else ['']
    return {
        'em': max(float(normalize_fa(prediction) == normalize_fa(ref)) for ref in candidates),
        'f1': max(token_f1(prediction, ref) for ref in candidates),
        'gold_no_answer': not bool(references),
        'pred_no_answer': normalize_fa(prediction) == '',
    }

def summarize_scores(records):
    if not records: raise ValueError('Cannot summarize empty predictions')
    a = np.array([x['em'] for x in records], dtype=np.float64)
    b = np.array([x['f1'] for x in records], dtype=np.float64)
    gold = np.array([x['gold_no_answer'] for x in records], dtype=bool)
    pred = np.array([x['pred_no_answer'] for x in records], dtype=bool)
    tp = int((gold & pred).sum()); fp = int((~gold & pred).sum())
    fn = int((gold & ~pred).sum()); tn = int((~gold & ~pred).sum())
    precision = tp / (tp+fp) if tp+fp else 0.0
    recall = tp / (tp+fn) if tp+fn else 0.0
    return {
        'n':len(records), 'em':float(a.mean()), 'f1':float(b.mean()),
        'answerable_f1':float(b[~gold].mean()) if (~gold).any() else None,
        'unanswerable_em':float(a[gold].mean()) if gold.any() else None,
        'no_answer_precision':precision, 'no_answer_recall':recall,
        'no_answer_f1':(2*precision*recall/(precision+recall) if precision+recall else 0.0),
        'no_answer_tp':tp, 'no_answer_fp':fp, 'no_answer_fn':fn,
        'no_answer_tn':tn,
    }

# Simple deterministic tests (including the repeated-token failure in the original code).
assert normalize_fa('كِتاب، ۱۲۳') == normalize_fa('کتاب 123')
assert token_f1('الف الف', 'الف') == 2/3
assert token_f1('', '') == 1.0
assert token_f1('یک پاسخ', '') == 0.0
assert score_example(NO_ANSWER_TEXT, [])['f1'] == 1.0
assert score_example('تهران', ['شیراز','تهران'])['em'] == 1.0
print('Metric tests passed.')

Metric tests passed.


## 6 — Baseline evaluation: unmodified Gemma 3 4B

This may require substantial Colab time on all 938 held-out examples. Predictions are written to Drive as JSONL so the data can be inspected independently. **Do not run training before this cell**; the baseline must be unmodified. The decoding setup is deterministic and shared with the later adapter evaluation.

In [8]:
def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def evaluate_model(active_model, name, eval_ds):
    active_model.eval()
    rows = []
    start = time.time()
    truncated_count = 0
    for i, row in enumerate(eval_ds):
        prefix, truncated_context, truncated = build_prompt(row)
        truncated_count += int(truncated)
        ids = tokenizer(prefix, return_tensors='pt', add_special_tokens=False)
        ids = {key: val.to('cuda') for key, val in ids.items()}
        # Same deterministic generation configuration for both systems.
        with torch.inference_mode():
            result = active_model.generate(
                **ids, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
            )
        # Decode only the new completion tokens, not the input prompt.
        new_ids = result[0][ids['input_ids'].shape[-1]:]
        prediction = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        refs = answer_texts(row)
        metrics = score_example(prediction, refs)
        rows.append({
            'id':str(row.get('id',i)), 'question':row['question'],
            'references':refs, 'prediction':prediction,
            'context_was_truncated':bool(truncated), **metrics,
        })
        if (i+1) % 50 == 0 or i+1 == len(eval_ds):
            print(f'{name}: {i+1}/{len(eval_ds)}')
        del ids, result, new_ids
    scores = summarize_scores(rows)
    scores['elapsed_seconds'] = round(time.time()-start, 2)
    scores['truncated_contexts'] = truncated_count
    scores['truncation_fraction'] = truncated_count/len(eval_ds)
    scores['model_id'] = MODEL_ID
    scores['model_revision'] = MODEL_REVISION
    scores['dataset_id'] = DATASET_ID
    scores['dataset_revision'] = DATASET_REVISION
    scores['official_test_split'] = OFFICIAL_TEST_SPLIT
    scores['smoke_test'] = bool(SMOKE_TEST)
    scores['generation'] = {'do_sample':False,'max_new_tokens':MAX_NEW_TOKENS,
                            'max_context_tokens':MAX_CONTEXT_TOKENS,
                            'max_seq_length':MAX_SEQ_LENGTH}
    write_jsonl(BASE_OUTPUT / f'{name}.predictions.jsonl', rows)
    write_json(BASE_OUTPUT / f'{name}.metrics.json', scores)
    return rows, scores

baseline_rows, baseline_scores = evaluate_model(model, 'baseline', official_test)
print(json.dumps(baseline_scores, ensure_ascii=False, indent=2))
print('Baseline saved to:', BASE_OUTPUT)

baseline: 50/930
baseline: 100/930
baseline: 150/930
baseline: 200/930
baseline: 250/930
baseline: 300/930
baseline: 350/930
baseline: 400/930
baseline: 450/930
baseline: 500/930
baseline: 550/930
baseline: 600/930
baseline: 650/930
baseline: 700/930
baseline: 750/930
baseline: 800/930
baseline: 850/930
baseline: 900/930
baseline: 930/930
{
  "n": 930,
  "em": 0.5344086021505376,
  "f1": 0.7241492698586394,
  "answerable_f1": 0.8010120137765508,
  "unanswerable_em": 0.5448028673835126,
  "no_answer_precision": 0.9743589743589743,
  "no_answer_recall": 0.5448028673835126,
  "no_answer_f1": 0.6988505747126438,
  "no_answer_tp": 152,
  "no_answer_fp": 4,
  "no_answer_fn": 127,
  "no_answer_tn": 647,
  "elapsed_seconds": 1306.93,
  "truncated_contexts": 0,
  "truncation_fraction": 0.0,
  "model_id": "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
  "model_revision": "316726ca0bd24aa323bfaf86e8a379ee1176d1fe",
  "dataset_id": "SajjadAyoubi/persian_qa",
  "dataset_revision": "5a314e73c690b159983b2

## 7 — Add LoRA to the same checkpoint and fine-tune

The base model weights stay quantized and frozen; only LoRA adapters are trained. This is a memory-conscious pilot for a single T4. We **do not** silently change this into a Gemma 2 or DeepSeek experiment. If you hit OOM: reduce `MAX_SEQ_LENGTH` and `MAX_CONTEXT_TOKENS` *in the configuration cell and rerun baseline + training + evaluation*, so the comparison remains controlled.

In [9]:
# Precompute tokenized answer-only training records before activating LoRA.
processed = []
skipped_long_target = 0
for ex in train_data:
    item = training_item(ex)
    if item is None:
        skipped_long_target += 1
    else:
        processed.append(item)
print('Usable training examples:', len(processed), 'skipped (long target / bounds):', skipped_long_target)
assert len(processed) > 0
train_token_ds = Dataset.from_list(processed)

def collate(batch):
    pad = tokenizer.pad_token_id
    longest = max(len(x['input_ids']) for x in batch)
    # Pad to multiple of eight for tensor cores, but never exceed sequence cap.
    width = min(MAX_SEQ_LENGTH, ((longest+7)//8)*8)
    tensors = {}
    for field, fill in [('input_ids',pad),('attention_mask',0),('labels',-100)]:
        tensors[field] = torch.tensor(
            [x[field] + [fill] * (width-len(x[field])) for x in batch],
            dtype=torch.long,
        )
    assert int((tensors['labels'] != -100).sum()) > 0
    return tensors

assert (collate([processed[0]])['labels'] != -100).any()
print('Training collator smoke check passed.')

Usable training examples: 8156 skipped (long target / bounds): 42
Training collator smoke check passed.


In [10]:
from transformers import Trainer, TrainingArguments

model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f'LoRA trainable parameters: {trainable:,} / {all_params:,} '
      f'({100*trainable/all_params:.3f}%)')

arguments = TrainingArguments(
    output_dir=str(BASE_OUTPUT / 'checkpoints'),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=0.03,
    lr_scheduler_type='cosine',
    weight_decay=0.05,
    fp16=True,
    bf16=False,
    optim='adamw_8bit',
    logging_steps=10,
    save_strategy='steps',
    save_steps=min(100, MAX_STEPS),
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    gradient_checkpointing=False,  # Already enabled by Unsloth.
    dataloader_num_workers=0,
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=train_token_ds,
    data_collator=collate,
)
train_result = trainer.train()
print('Training metrics:', train_result.metrics)
ADAPTER_DIR = BASE_OUTPUT / 'lora_adapter'
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
write_json(BASE_OUTPUT/'training_manifest.json', {
    'model_id':MODEL_ID, 'model_revision':MODEL_REVISION,
    'dataset_id':DATASET_ID, 'dataset_revision':DATASET_REVISION,
    'official_test_split':OFFICIAL_TEST_SPLIT,
    'seed':SEED, 'max_steps':MAX_STEPS,
    'training_rows':len(processed), 'skipped_long_targets':skipped_long_target,
    'batch_size':BATCH_SIZE, 'gradient_accumulation_steps':GRAD_ACCUMULATION,
    'lora_rank':LORA_R, 'lora_alpha':LORA_ALPHA,
    'learning_rate':LEARNING_RATE, 'max_seq_length':MAX_SEQ_LENGTH,
    'max_context_tokens':MAX_CONTEXT_TOKENS, 'max_new_tokens':MAX_NEW_TOKENS,
    'trainable_parameters':trainable, 'all_parameters':all_params,
    'train_metrics':train_result.metrics, 'smoke_test':SMOKE_TEST,
})
print('LoRA adapter saved:', ADAPTER_DIR)

LoRA trainable parameters: 32,788,480 / 2,972,340,592 (1.103%)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,156 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.533124
20,0.439927
30,0.260813
40,0.256305
50,0.202156
60,0.267109
70,0.225245
80,0.224210
90,0.271574
100,0.237853


Training metrics: {'train_runtime': 1565.5077, 'train_samples_per_second': 1.278, 'train_steps_per_second': 0.16, 'total_flos': 1.82095911518976e+16, 'train_loss': 0.27062832641601564, 'epoch': 0.24521824423737126}
LoRA adapter saved: /content/drive/MyDrive/Gemma3_4B_PersianQA/lora_adapter


## 8 — Fine-tuned model evaluation (same official held-out split)

Identical test questions, prompt construction, decoding and metrics. The adapter is active on the same in-memory quantized checkpoint; **no base-model re-download is required**.

In [11]:
# Trainer may leave the model in training mode; evaluate_model switches to eval mode.
finetuned_rows, finetuned_scores = evaluate_model(model, 'finetuned', official_test)
assert len(baseline_rows) == len(finetuned_rows) == len(official_test)
assert [x['id'] for x in baseline_rows] == [x['id'] for x in finetuned_rows]
assert not any(x['references'] != y['references'] for x,y in zip(baseline_rows,finetuned_rows))
print(json.dumps(finetuned_scores, ensure_ascii=False, indent=2))

finetuned: 50/930
finetuned: 100/930
finetuned: 150/930
finetuned: 200/930
finetuned: 250/930
finetuned: 300/930
finetuned: 350/930
finetuned: 400/930
finetuned: 450/930
finetuned: 500/930
finetuned: 550/930
finetuned: 600/930
finetuned: 650/930
finetuned: 700/930
finetuned: 750/930
finetuned: 800/930
finetuned: 850/930
finetuned: 900/930
finetuned: 930/930
{
  "n": 930,
  "em": 0.621505376344086,
  "f1": 0.8124203511647292,
  "answerable_f1": 0.7704315308497667,
  "unanswerable_em": 0.910394265232975,
  "no_answer_precision": 0.9136690647482014,
  "no_answer_recall": 0.910394265232975,
  "no_answer_f1": 0.9120287253141832,
  "no_answer_tp": 254,
  "no_answer_fp": 24,
  "no_answer_fn": 25,
  "no_answer_tn": 627,
  "elapsed_seconds": 2032.52,
  "truncated_contexts": 0,
  "truncation_fraction": 0.0,
  "model_id": "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
  "model_revision": "316726ca0bd24aa323bfaf86e8a379ee1176d1fe",
  "dataset_id": "SajjadAyoubi/persian_qa",
  "dataset_revision": "5a314

## 9 — Paired analysis, uncertainty and report

Report score changes in **percentage points**, not an unsupported generic claim of improvement. Confidence intervals use paired resampling of the same test examples; the bootstrap is illustrative and cannot correct dataset artifacts or prompt mismatches. No test-based model selection is performed.

In [12]:
def paired_bootstrap(base, tuned, field='f1', repeats=BOOTSTRAP_REPEATS, seed=SEED):
    a = np.asarray([row[field] for row in base], dtype=np.float64)
    b = np.asarray([row[field] for row in tuned], dtype=np.float64)
    assert len(a) == len(b) and len(a)>1
    difference = b-a
    generator = np.random.default_rng(seed)
    estimates = np.empty(repeats, dtype=np.float64)
    for k in range(repeats):
        sample_idx = generator.integers(0,len(difference),size=len(difference))
        estimates[k] = difference[sample_idx].mean()
    lo,hi = np.quantile(estimates,[0.025,0.975])
    return {'delta':float(difference.mean()), 'ci95':[float(lo),float(hi)]}

comparison = {
    'model_id':MODEL_ID, 'model_revision':MODEL_REVISION,
    'dataset_id':DATASET_ID, 'dataset_revision':DATASET_REVISION,
    'n':len(official_test), 'official_test_split':OFFICIAL_TEST_SPLIT,
    'smoke_test':bool(SMOKE_TEST),
    'baseline':baseline_scores, 'fine_tuned':finetuned_scores,
    'paired_f1':paired_bootstrap(baseline_rows,finetuned_rows,'f1'),
    'paired_em':paired_bootstrap(baseline_rows,finetuned_rows,'em'),
    'training_steps':MAX_STEPS,
    'note':('SMOKE TEST: not a full-test benchmark' if SMOKE_TEST else
            'Entire official test split used; training used a bounded T4 pilot'),
}
write_json(BASE_OUTPUT/'comparison.json',comparison)

changed = []
for before, after in zip(baseline_rows,finetuned_rows):
    changed.append({
        'id': before['id'], 'question':before['question'],
        'references':before['references'],
        'baseline_prediction':before['prediction'],
        'finetuned_prediction':after['prediction'],
        'baseline_f1':before['f1'], 'finetuned_f1':after['f1'],
        'delta_f1':after['f1']-before['f1'],
    })
write_jsonl(BASE_OUTPUT/'per_example_comparison.jsonl',changed)

f1 = comparison['paired_f1']; em = comparison['paired_em']
report = f"""# Gemma 3 4B PersianQA controlled experiment

- Status: {'SMOKE TEST; not research evidence' if SMOKE_TEST else 'FULL HELD-OUT TEST'}
- Base checkpoint: `{MODEL_ID}` (`{MODEL_REVISION}`)
- Dataset: `{DATASET_ID}` (`{DATASET_REVISION}`)
- Official held-out split: `{OFFICIAL_TEST_SPLIT}`, N={len(official_test)}
- LoRA training: {MAX_STEPS} optimization steps, r={LORA_R}, alpha={LORA_ALPHA}, lr={LEARNING_RATE}
- Generation: greedy, max_new_tokens={MAX_NEW_TOKENS}; maximum context tokens={MAX_CONTEXT_TOKENS}

| Metric | Untuned Gemma 3 4B IT | Gemma 3 4B IT + LoRA | Difference (percentage points) |
|---|---:|---:|---:|
| Exact Match | {100*baseline_scores['em']:.2f}% | {100*finetuned_scores['em']:.2f}% | {100*em['delta']:+.2f} |
| Token F1 | {100*baseline_scores['f1']:.2f}% | {100*finetuned_scores['f1']:.2f}% | {100*f1['delta']:+.2f} |
| No-answer F1 | {100*baseline_scores['no_answer_f1']:.2f}% | {100*finetuned_scores['no_answer_f1']:.2f}% | {100*(finetuned_scores['no_answer_f1']-baseline_scores['no_answer_f1']):+.2f} |

Paired bootstrap 95% CI for delta F1: [{100*f1['ci95'][0]:+.2f}, {100*f1['ci95'][1]:+.2f}] pp.
Paired bootstrap 95% CI for delta EM: [{100*em['ci95'][0]:+.2f}, {100*em['ci95'][1]:+.2f}] pp.

## Limitations
- Comparison is within the Gemma 3 4B 4-bit IT checkpoint, not the old Gemma 2 9B project.
- Contexts may be shortened; {baseline_scores['truncated_contexts']}/{len(official_test)} prompts were truncated.
- {MAX_STEPS} optimization steps are a bounded T4 pilot, not proof of convergence.
- Single-seed model adaptation; inference is greedy and deterministic.
- Normalized EM/F1 can differ from the dataset authors' official metric implementation.
- No GPU training score can be claimed unless this notebook actually completed the experiments.
"""
(BASE_OUTPUT/'report.md').write_text(report,encoding='utf-8')
print(report)
print('RESULTS FOLDER:',BASE_OUTPUT)

# Gemma 3 4B PersianQA controlled experiment

- Status: FULL HELD-OUT TEST
- Base checkpoint: `unsloth/gemma-3-4b-it-unsloth-bnb-4bit` (`316726ca0bd24aa323bfaf86e8a379ee1176d1fe`)
- Dataset: `SajjadAyoubi/persian_qa` (`5a314e73c690b159983b2e45b9d4c0500a80cfd0`)
- Official held-out split: `validation`, N=930
- LoRA training: 250 optimization steps, r=16, alpha=32, lr=0.0001
- Generation: greedy, max_new_tokens=72; maximum context tokens=560

| Metric | Untuned Gemma 3 4B IT | Gemma 3 4B IT + LoRA | Difference (percentage points) |
|---|---:|---:|---:|
| Exact Match | 53.44% | 62.15% | +8.71 |
| Token F1 | 72.41% | 81.24% | +8.83 |
| No-answer F1 | 69.89% | 91.20% | +21.32 |

Paired bootstrap 95% CI for delta F1: [+6.36, +11.34] pp.  
Paired bootstrap 95% CI for delta EM: [+5.37, +12.15] pp.

## Limitations
- Comparison is within the Gemma 3 4B 4-bit IT checkpoint, not the old Gemma 2 9B project.
- Contexts may be shortened; 0/930 prompts were truncated.
- 250 optimization steps are a bo

## 10 — Results files and research interpretation

Saved in your Drive folder (or `/content/gemma3_persianqa_results` if Drive was unavailable):

- `baseline.predictions.jsonl`, `baseline.metrics.json`
- `finetuned.predictions.jsonl`, `finetuned.metrics.json`
- `lora_adapter/`, `checkpoints/`, `training_manifest.json`
- `comparison.json`, `per_example_comparison.jsonl`, `report.md`

**Use `comparison.json` and `report.md` to establish the measured result; do not reuse Gemma 2's old 0.49 F1.** A convincing CV claim should mention the actual model, full-test size, metric difference and training budget. If the 95% CI spans zero, avoid claiming a statistically supported improvement.

**Reproducibility caution:** because Colab environments and upstream dependencies change, this notebook has been statically validated and unit-tested for its metric logic, but **not end-to-end executed on a T4 here**. If dependency/API changes cause an error, record the version/traceback and correct the smallest affected cell before using the scores.

**Sources:** [Gemma model sizes](https://ai.google.dev/gemma/docs/get_started) · [Unsloth Gemma 3 T4 notes](https://unsloth.ai/docs/models/tutorials/gemma-3-how-to-run-and-fine-tune) · [PersianQA dataset](https://huggingface.co/datasets/SajjadAyoubi/persian_qa).